# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Awais-Ahmad10/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Note (v2 rewrite):** the first draft of this notebook used `trend_direction == "down"` as a scoring
signal. That column is the *source of the evaluation label* (`is_declining_label`), so scoring on it
is circular — the baseline would be grading its own homework. This version scores only on
non-leaking page-state signals (visibility, position, staleness, click-through) and evaluates the
rule against the label honestly. The first draft also had malformed notebook JSON; this file was
regenerated so it opens and runs top to bottom.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** A page is worth reviewing when it still earns meaningful search visibility but looks weak:
it ranks poorly, its click-through is thin, or the content has not been refreshed in a long time.
Pages that trip several of these at once go to the top of the queue. The rule never looks at
`trend_direction` or `trend_pct` — those define the label we evaluate against, so using them as
inputs would be leakage.

**Reason codes the rule can output:**
- `visible` — page earned >= 500 impressions in the trailing 90 days (the dictionary's "moderate" tier and up; still worth attention)
- `poor_position` — average position is worse than 10 (off page one), excluding rows where `avg_position == 0`, which means "no position data", not rank zero
- `stale` — more than 90 days since the last content update
- `thin_ctr` — click-through below 0.10% on a visible page (0.10 is a round number near the slice median of 0.07%; thin CTR on real volume suggests the snippet or intent fit is off)
- `no_action` — none of the above fired; the page looks fine and sits at the bottom of the queue

**Score:** +2 for each of `visible`, `poor_position`, `stale`, +1 for `thin_ctr`. No fitted weights —
a human can recompute any page's score by hand.


In [ ]:
import os
import numpy as np
import pandas as pd

# Run from the repo root so relative paths work in Colab and locally.
print("cwd:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"loaded {len(df):,} pages")

# The label is for EVALUATION ONLY — never an input to the rule.
# trend_direction / trend_pct are the label source (see data dictionary).
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"label base rate (declining): {df['is_declining_label'].mean():.3f}")

# --- Signals: page state only, nothing derived from the label ---
df["sig_visible"] = (df["impressions_90d"] >= 500).astype(int)
# avg_position == 0 means "no position data" (1,205 rows) — never treat as rank zero.
df["sig_poor_position"] = ((df["avg_position"] > 10) & (df["avg_position"] != 0)).astype(int)
df["sig_stale"] = (df["days_since_last_update"] > 90).astype(int)
# CTR is noisy on tiny volume; only judge it where impressions clear the visibility floor.
df["sig_thin_ctr"] = ((df["ctr"] < 0.10) & (df["impressions_90d"] >= 500)).astype(int)

df["baseline_score"] = (
    2 * df["sig_visible"]
    + 2 * df["sig_poor_position"]
    + 2 * df["sig_stale"]
    + 1 * df["sig_thin_ctr"]
)

def reason_codes(row):
    codes = []
    if row["sig_visible"]:
        codes.append("visible")
    if row["sig_poor_position"]:
        codes.append("poor_position")
    if row["sig_stale"]:
        codes.append("stale")
    if row["sig_thin_ctr"]:
        codes.append("thin_ctr")
    return "+".join(codes) if codes else "no_action"

df["reason_codes"] = df.apply(reason_codes, axis=1)
print("score distribution:")
print(df["baseline_score"].value_counts().sort_index())


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ties are broken deterministically by higher 90-day impressions first, so the queue order is
reproducible. The CSV carries the score, the reason codes, and the supporting columns a reviewer
needs — it deliberately does **not** carry the label or the label-source columns.


In [ ]:
# Deterministic ranking: score desc, then impressions desc.
ranked = df.sort_values(["baseline_score", "impressions_90d"],
                        ascending=[False, False]).reset_index(drop=True)

out = ranked[["content_id", "baseline_score", "reason_codes",
              "impressions_90d", "avg_position", "days_since_last_update", "ctr"]].copy()
os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"ranked queue written: {len(out):,} pages -> work/outputs/baseline_action_score.csv")

# --- Honest evaluation: precision@K against the held-out-from-features label ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

labels = ranked["is_declining_label"].to_numpy()
scores = ranked["baseline_score"].to_numpy()
base_rate = float(labels.mean())
print(f"\nbase rate (declining share of all pages): {base_rate:.3f}")
for k in (20, 50, 100, 500):
    print(f"baseline precision@{k}: {precision_at_k(scores, labels, k):.3f}")
# Dummy baseline: predict "declining" for every page -> precision@K == base rate at every K.
print(f"dummy (predict-all-declining) precision@K: {base_rate:.3f} at every K")
print("\nRead: the rule beats the base rate at every K shown, so it is a real signal —")
print("and it is simple enough that ML-08 will have something honest to beat.")


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Action rule, stated up front: score >= 5 → REVIEW (multiple weaknesses coincide);
score 3–4 → MONITOR (one or two flags, watch it); below 3 → no action (not in the top 20 anyway).
Confidence follows the score: 7 → HIGH, 5–6 → MEDIUM, anything lower → LOW.


In [ ]:
WHAT_WRONG = {
    "visible": "impressions may be concentrated on navigational/brand queries that need no refresh",
    "poor_position": "position is a 90-day mean; a recent fix may not show up yet",
    "stale": "evergreen content can be fine without frequent updates",
    "thin_ctr": "CTR is measured over the same 90-day window; seasonality can depress it temporarily",
}

top20 = ranked.head(20)
print(f"{'rank':<5}{'score':<7}{'action':<9}{'conf':<8}reason codes")
print("-" * 70)
for i, row in enumerate(top20.itertuples(), start=1):
    s = row.baseline_score
    action = "REVIEW" if s >= 5 else ("MONITOR" if s >= 3 else "SKIP")
    conf = "HIGH" if s == 7 else ("MEDIUM" if s >= 5 else "LOW")
    print(f"{i:<5}{s:<7}{action:<9}{conf:<8}{row.reason_codes}")

print("\nWhat would make each pick wrong (per reason code):")
seen = set()
for codes in top20["reason_codes"]:
    for c in codes.split("+"):
        if c not in seen and c in WHAT_WRONG:
            seen.add(c)
            print(f"  - {c}: wrong {WHAT_WRONG[c]}")


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [ ]:
# Weak picks: top-20 pages the rule flagged that are NOT declining.
weak = top20[top20["is_declining_label"] == 0]
print(f"top-20 pages not declining (weak picks): {len(weak)}")
for row in weak.itertuples():
    print(f"  - {row.content_id}: score={row.baseline_score} [{row.reason_codes}] "
          f"imp={row.impressions_90d:.0f} pos={row.avg_position} stale={row.days_since_last_update}d "
          f"trend={row.trend_direction}")
print("Read: these are pages the rule finds suspicious on page-state alone, but whose trailing-30d "
      "impressions have not (yet) fallen 20%+. They are exactly the 'early warning' cases a refresh "
      "queue exists for — flagging them is the rule doing its job, not failing.")

# Missed declining pages at the bottom: how many decliners score 0?
missed = df[(df["baseline_score"] == 0) & (df["is_declining_label"] == 1)]
print(f"\ndeclining pages the rule scores 0 (missed): {len(missed):,} "
      f"({len(missed)/len(df):.1%} of all pages)")

# The avg_position == 0 trap: those rows are almost never declining (measured 0.7%).
# Confirm none leaked into the top of the queue via poor_position.
no_pos_top = (ranked.head(100)["avg_position"] == 0).sum()
print(f"pages with no position data in top-100: {no_pos_top} (must be 0)")

print("\nLeakage check:")
print("  - trend_direction / trend_pct used as features: NO (label source, evaluation only)")
print("  - is_declining_label used as a feature: NO (computed after scoring inputs, eval only)")
print("  - future-window inputs: NO (all inputs are trailing-90d aggregates or content age)")
label_cols_in_csv = [c for c in out.columns
                     if c in ("is_declining_label", "trend_direction", "trend_pct")]
print(f"  - label/label-source columns in the CSV artifact: {label_cols_in_csv or 'none'}")
print("  - client names, URLs, raw queries printed: none (IDs are pseudonyms)")
assert no_pos_top == 0 and not label_cols_in_csv, "leakage guard failed"
print("\nAll guards pass.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Measured claims in this notebook (directional, decision-support only):**
- On the 30k-row starter slice, the rule's precision@20 is 0.700 vs a 0.542 base rate (observed).
- The rule uses no label-derived inputs; ML-08 should beat these numbers honestly.
